# LangChain Complete Tutorial

A comprehensive notebook covering all key topics in the LangChain framework.

## Table of Contents
1. LangChain Overview
2. Building Blocks (Chat Models, Prompts, Parsers)
3. Chains (LLMChain, SequentialChain, RouterChain)
4. Memory, Tools, and Agents
5. Patterns and Best Practices
6. Practical Examples

## 1. LangChain Overview

LangChain is an open-source framework for building applications with large language models (LLMs).

### Key Features:
- **Abstraction**: Simplified interfaces for LLM interactions
- **Composability**: Chain components together
- **Tool Ecosystem**: Built-in integrations for tools, vector databases
- **Memory**: Built-in conversation memory
- **Agents**: Autonomous agents with tool usage
- **Production Features**: Debugging, monitoring, evaluation

In [ ]:
# Installation
!pip install langchain langchain-openai langchain-community

### LangChain vs LangGraph

| Aspect | LangChain | LangGraph |
|--------|-----------|-----------|
| Structure | Linear chains | Cyclic graphs |
| Use Case | Simple workflows | Complex, stateful agents |
| Control Flow | Sequential | Conditional, branching |
| State Management | Basic | Sophisticated |
| Best for | Prototypes | Production agents |

## 2. Building Blocks

### 2.1 Chat Models

Chat models are the core of LangChain applications. Supported providers include OpenAI, Anthropic, Azure OpenAI, Google Vertex AI, and HuggingFace.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

# OpenAI Chat
llm = ChatOpenAI(model="gpt-4", temperature=0.7, max_tokens=1000)

# Anthropic Chat
# claude = ChatAnthropic(model="claude-3-opus")

### Key Parameters:
- **temperature**: Controls randomness (0-2)
- **max_tokens**: Maximum tokens in response
- **model_name**: Which model to use
- **streaming**: Enable streaming responses

### 2.2 Prompt Templates

Prompt templates create reusable, parameterized prompts.

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# String Prompt Template
string_template = PromptTemplate.from_template("Tell me a {adjective} joke about {topic}")

# Chat Prompt Template
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role} assistant."),
    ("human", "{user_input}")
])

# Format prompts
formatted_string = string_template.format(adjective="funny", topic="AI")
formatted_chat = chat_template.format(role="helpful", user_input="What is AI?")

### 2.3 Output Parsers

Output parsers transform LLM responses into usable formats.

In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser
from pydantic import BaseModel

# String Parser
str_parser = StrOutputParser()

# JSON Parser
json_parser = JsonOutputParser()

# Pydantic Parser
class Recipe(BaseModel):
    name: str
    ingredients: list[str]
    instructions: list[str]
    prep_time_minutes: int

pydantic_parser = PydanticOutputParser(pydantic_object=Recipe)

### 2.4 Caching

Caching reduces costs and improves speed by storing LLM responses.

In [ ]:
from langchain_community.cache import InMemoryCache
from langchain_core.globals import set_llm_cache

# Enable in-memory caching
set_llm_cache(InMemoryCache())

### 2.5 Streaming

Streaming enables real-time response display.

In [ ]:
# Streaming response (token-by-token)
# for chunk in llm.stream("Tell me a story"):
#     print(chunk.content, end="")

## 3. Chains (LCEL)

Chains combine multiple components using the LangChain Expression Language (LCEL).

In [ ]:
# Simple LCEL Chain
chain = string_template | llm | str_parser
result = chain.invoke({"adjective": "funny", "topic": "AI"})
print(result)

### 3.1 LLMChain

LLMChain is the basic chain combining prompt + LLM.

In [ ]:
from langchain.chains import LLMChain

# LLMChain - Basic prompt + LLM
llm_chain = LLMChain(llm=llm, prompt=string_template, output_key="joke")
result = llm_chain.invoke({"adjective": "hilarious", "topic": "data science"})
print(result["joke"])

### 3.2 SequentialChain

SequentialChain runs multiple chains in sequence.

In [ ]:
from langchain.chains import SequentialChain, LLMChain
from langchain.prompts import PromptTemplate

# Chain 1: Generate facts
facts_prompt = PromptTemplate(
    input_variables=["topic"],
    template="List 3 interesting facts about {topic}"
)
facts_chain = LLMChain(llm=llm, prompt=facts_prompt, output_key="facts")

# Chain 2: Summarize facts
summary_prompt = PromptTemplate(
    input_variables=["facts"],
    template="Summarize these facts into one sentence: {facts}"
)
summary_chain = LLMChain(llm=llm, prompt=summary_prompt, output_key="summary")

# Sequential Chain
seq_chain = SequentialChain(
    chains=[facts_chain, summary_chain],
    input_variables=["topic"],
    output_variables=["facts", "summary"],
)

result = seq_chain.invoke({"topic": "artificial intelligence"})
print("Facts:", result["facts"])
print("Summary:", result["summary"])

### 3.3 RouterChain

RouterChain routes to different chains based on input.

In [ ]:
from langchain.chains import RouterChain
from langchain.chains.llm import LLMChain

# Define destination chains
physics_prompt = PromptTemplate(template="Explain {topic} in physics")
math_prompt = PromptTemplate(template="Explain {topic} in mathematics")

physics_chain = LLMChain(llm=llm, prompt=physics_prompt)
math_chain = LLMChain(llm=llm, prompt=math_prompt)

# Create router
default_chain = LLMChain(llm=llm, prompt=PromptTemplate(template="{topic}"))

router_chain = RouterChain(
    default_chain=default_chain,
    destination_chains={
        "physics": physics_chain,
        "math": math_chain
    }
)

## 4. Memory Types

Memory allows chains to maintain state between interactions.

In [ ]:
from langchain.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory,
    ConversationTokenBufferMemory
)

# Buffer Memory - Store all messages
buffer_memory = ConversationBufferMemory(
    return_messages=True,
    output_key="text",
    input_key="input"
)

# Window Memory - Last K messages
window_memory = ConversationBufferWindowMemory(k=5, return_messages=True)

# Summary Memory - Summarized history
summary_memory = ConversationSummaryMemory(llm=llm)

### Memory Types Summary:
- **ConversationBufferMemory**: Store all messages
- **ConversationBufferWindowMemory**: Last K messages only
- **ConversationSummaryMemory**: Summarized conversation history
- **ConversationTokenBufferMemory**: By token limit
- **VectorStoreMemory**: Semantic retrieval from history

## 5. Tools and Agents

### 5.1 Creating Tools

Tools extend agent capabilities with custom functions.

In [ ]:
from langchain.tools import tool

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

### Agent Loop:
1. Receive input
2. Decide action
3. Execute tool
4. Observe result
5. Repeat until done

## 6. Patterns and Best Practices

### 6.1 Error Handling

In [ ]:
# Error handling best practices:
- Try/except: Wrap LLM calls
- Retry logic: Use retry callbacks
- Fallback chains: Alternate on failure
- Timeout: Set max execution time
- Circuit breaker: Prevent cascade failures

### 6.2 Production Optimization

In [ ]:
# Optimization strategies:
- Caching: Cache LLM responses
- Prompt optimization: Reduce tokens
- Smaller models: Use cheaper models when possible
- Batch processing: Group requests
- Memory selection: Choose appropriate memory type

### 6.3 Runnable Interface

The standard interface for all LangChain components.

In [ ]:
# Runnable methods:
- .invoke() - Synchronous single call
- .batch() - Synchronous batch
- .stream() - Streaming
- .ainvoke() - Async single call
- .abatch() - Async batch

# Composition with pipe operator:
chain = prompt | llm | output_parser

## 7. Practical Examples

### Example 1: Basic Chatbot with Memory

In [ ]:
import os
from langchain_openai import ChatOpenAI

# Set your OpenAI API key as an environment variable
# Replace "YOUR_OPENAI_API_KEY" with your actual key
os.environ["OPENAI_API_KEY"] = "sk-" # <--- IMPORTANT: Replace this with your actual API key

llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

# Create conversation chain with memory
conv = ConversationChain(
    llm=llm,
    memory=ConversationBufferMemory()
)

# Interact
response = conv.predict(input="Hi! I am John.")
print(response)

response = conv.predict(input="What is my name?")
print(response)

### Example 2: Tool-using Agent

In [ ]:
# Create agent with calculator tool
math_agent = initialize_agent(
    tools=[calculate],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION
)

# Use agent
result = math_agent.invoke("What is 5 + 3?")
print(result)

### Example 3: RAG Chain (Question Answering)

In [ ]:
# RAG (Retrieval Augmented Generation)
# from langchain.chains import RetrievalQA
# from langchain.vectorstores import Chroma
# from langchain.embeddings import OpenAIEmbeddings

# Create vector store
# vectorstore = Chroma.from_documents(documents, embedding=OpenAIEmbeddings())
# retriever = vectorstore.as_retriever()

# Create QA chain
# qa_chain = RetrievalQA.from_chain_type(
#     llm=llm,
#     chain_type="stuff",
#     retriever=retriever,
#     return_source_documents=True
# )

## 8. Summary

Key LangChain concepts covered:
1. **Overview**: Framework, components, ecosystem
2. **Building Blocks**: Models, prompts, parsers, caching, streaming
3. **Chains**: LLMChain, SequentialChain, RouterChain, LCEL
4. **Memory**: Buffer, Window, Summary, Token types
5. **Tools & Agents**: Tool creation, agent types, agent loop
6. **Best Practices**: Error handling, optimization, production